In [4]:
import sys
sys.path.append('..')

In [6]:
import json
import numpy as np
import pandas as pd
import re
from collections import defaultdict
import yaml
import random
from sklearn.model_selection import train_test_split
from datasets import Dataset
from data.industrial.templates import *
import os

In [7]:
p_desc = 0.4
seed_no = 42
if sys.argv[1] != '-f':
    p_desc = float(sys.argv[1])
    seed_no = int(sys.argv[2])

In [8]:
print(f'p_desc: {p_desc}, seed_no: {seed_no}')

p_desc: 0.4, seed_no: 42


In [9]:
random.seed(seed_no)

In [10]:
all_dicts = {}

In [11]:
with open('../data/industrial/synthetic_desc/other_descs.json', 'r') as f:
    other_descs = json.load(f)

## 1. Equipment category to equipments (E2CAT)

In [12]:
instruction = "Given an equipment category what are the most relevant equipments?"

In [13]:
fname = '../data/industrial/iso_equipment_category.json'

In [14]:
with open(fname, 'r') as f:
    data = json.load(f)

In [15]:
result_dict = {}

In [16]:
for item in data:
    instruction = random.choice(equipment_category_to_equipments)
    query = f'Instruct: {instruction}\nQuery: Equipment category: {item["Equipment Category"]}'
    if random.random() < p_desc:
        query += f', Description: {other_descs["eq_cat"][item["Equipment Category"]]}'
    result_dict[query] = [f"Equipment: {x}" for x in item['Equipment Class']]

In [17]:
key = list(result_dict.keys())[0]
key, result_dict[key]

('Instruct: What equipment is most pertinent to a particular equipment category?\nQuery: Equipment category: Drilling, Description: Drilling equipment includes machines and tools used to create holes in various materials, such as rock, metal, and wood, by rotating a drill bit at high speed.',
 ['Equipment: Cementing equipment',
  'Equipment: Choke and manifolds',
  'Equipment: Crown and travelling blocks',
  'Equipment: Derrick',
  'Equipment: Diverters',
  'Equipment: Drawworks',
  'Equipment: Drilling and completion risers',
  'Equipment: Drill strings',
  'Equipment: Mud-treatment equipment',
  'Equipment: Pipe handling equipment',
  'Equipment: Riser compensators',
  'Equipment: String-motion compensators',
  'Equipment: Subsea blowout preventers (BOP)',
  'Equipment: Surface blowout preventers (BOP)',
  'Equipment: Top drives'])

In [18]:
all_dicts['eq_to_category'] = result_dict.copy()

In [19]:
len(np.unique(list(result_dict.keys()))), len(np.unique([val for item in result_dict.values() for val in item])), np.mean([len(val) for val in result_dict.values()])

(10, 107, 10.7)

## 2. Equipment class to equipment type (E2CLT)

In [23]:
fname = '../data/industrial/iso_equipment_class_type.json'

In [24]:
with open(fname, 'r') as f:
    data = json.load(f)

In [25]:
result_dict = {}

In [26]:
instruction = 'Given an equipment class what are the most relevant equipment types?'

In [27]:
for item in data:
    instruction = random.choice(equipment_class_to_type)
    query = f'Instruct: {instruction}\nQuery: Equipment Class: {item["Equipment Class"]}'
    if random.random() < p_desc:
        query += ', Description: ' + other_descs['eq_class'][item['Equipment Class']]
    result_dict[query] = [f'Equipment Type: {x}' for x in item["Equipment Type"]] 

In [28]:
query

'Instruct: Within a given equipment class, what types of equipment are most relevant?\nQuery: Equipment Class: Winches'

In [29]:
key = list(result_dict.keys())[0]
key, result_dict[key]

('Instruct: Given an equipment class, which equipment types are the most relevant?\nQuery: Equipment Class: Combustion engines',
 ['Equipment Type: Diesel engine', 'Equipment Type: Otto (gas) engine'])

In [30]:
all_dicts['eq_to_class_type'] = result_dict.copy()

In [31]:
len(np.unique(list(result_dict.keys()))), len(np.unique([val for item in result_dict.values() for val in item])), np.mean([len(val) for val in result_dict.values()])

(42, 156, 4.5)

## 3. Equipment subunit to unit (ES2U)

In [34]:
fname = '../data/industrial/iso_equipment_unit_subunit.json'
instruction = 'For a given equipment unit what are the component names and their groups that it consists of?'

In [35]:
with open(fname, 'r') as f:
    data = json.load(f)

In [36]:
result_dict = {}

In [37]:
for item in data:
    instruction = random.choice(equipment_subunit_to_unit)
    query = f'Instruct: {instruction}\nQuery: Equipment unit: {item["Equipment unit"]}'
    result_dict[query] = []
    for subunit in item['Subunit']:
        texts = []
        texts = [f'Component group: {subunit["name"]}, Component name: {component}' for component in subunit["Maintainable items"]]
        result_dict[query].extend(texts)

In [38]:
key = list(result_dict.keys())[0]
key, result_dict[key][:5]

('Instruct: For a specified equipment unit, what are the components and the groups they are classified under?\nQuery: Equipment unit: Combustion engines',
 ['Component group: Start system, Component name: Start energy (battery, air)',
  'Component group: Start system, Component name: Starting unit',
  'Component group: Start system, Component name: Start control',
  'Component group: Combustion engine unit, Component name: Air inlet',
  'Component group: Combustion engine unit, Component name: Ignition system'])

In [39]:
all_dicts['eq_subunit_to_unit'] = result_dict.copy()

In [40]:
len(np.unique(list(result_dict.keys()))), len(np.unique([val for item in result_dict.values() for val in item])), np.mean([len(val) for val in result_dict.values()])

(43, 1191, 33.13953488372093)

# 4. Failure Description -> Failure class (FM2CLS)

In [41]:
instruction = 'Given a failure description what is its failure class?'

In [43]:
fname = '../data/industrial/iso_failure_mode_metadata.json'

In [44]:
data = []
with open(fname, 'r') as f:
    for line in f.readlines():
        data.append(json.loads(line))

In [45]:
result_dict = {}

In [46]:
already_seen = set()
for item in data:
    for x in item['examples']:
        if x['sample'] in already_seen:
            continue
        already_seen.add(x['sample'])
        instruction = random.choice(failure_description_to_failure_class)
        query = f'Instruct: {instruction}\nQuery: Failure Description: {x["sample"]}'
        result_dict[query] = [f'Failure class: {item["description"]} ({item["label"]})']

In [47]:
key = list(result_dict.keys())[0]
key, result_dict[key]

('Instruct: Given a failure description, which failure mode class best fits it?\nQuery: Failure Description: False alarm, faulty instrument indication',
 ['Failure class: Abnormal instrument reading (AIR)'])

In [48]:
all_dicts['failure_desc_to_class'] = result_dict.copy()

In [49]:
len(np.unique(list(result_dict.keys()))), len(np.unique([val for item in result_dict.values() for val in item])), np.mean([len(val) for val in result_dict.values()])

(140, 62, 1.0)

## 5. Given asset_name (subsea_equipment) + failure_mode_class (internal leakage medium) -> component discovery ([Subsea production control"...]) (FM2CMP)

In [50]:
instruction = "Given an asset name and a failure mode class, what are the associated components?"

In [51]:
result_dict = {}
for item in data:
    for x in item['examples']:
        instruction = random.choice(asset_class_failure_mode_class_to_component)
        query = f'Instruct: {instruction}\nQuery: Asset name: {x["asset_name"]}, failure mode class: {item["description"]}'
        result_dict[query] = [f"Component: {component}" for component in x['components']]

In [52]:
key = list(result_dict.keys())[0]
key, result_dict[key]

('Instruct: In the context of an asset name and failure mode class, what components are affected?\nQuery: Asset name: drilling_equipment, failure mode class: Abnormal instrument reading',
 ['Component: Top drives',
  'Component: Subsea blowout preventers (BOP)',
  'Component: Surface blowout preventers (BOP)'])

In [53]:
all_dicts['asset_fm_to_components'] = result_dict.copy()

In [54]:
len(np.unique(list(result_dict.keys()))), len(np.unique([val for item in result_dict.values() for val in item])), np.mean([len(val) for val in result_dict.values()])

(254, 44, 2.673228346456693)

## 6. Component to failure mode (C2FM)

In [55]:
instruction = f"Given an asset component, what are possible failure modes it can experience?"

In [56]:
result_dict = defaultdict(list)

In [57]:
already_seen = set()
for item in data:
    for sample in item['examples']:
        for component in sample['components']:
            if component in already_seen:
                continue
            already_seen.add(component)
            instruction = random.choice(component_to_failure_mode)
            query = f"Instruct: {instruction}\nQuery: Component: {component}"
            if random.random() < p_desc:
                query += ', Description: ' + other_descs['component'][component]
            result_dict[query].append(f"Failure mode: {item['description']} ({item['label']})")

In [58]:
key = list(result_dict.keys())[0]
key, result_dict[key]

('Instruct: For a specified component, what are the possible failure modes it may experience?\nQuery: Component: Top drives',
 ['Failure mode: Abnormal instrument reading (AIR)'])

In [59]:
all_dicts['component_to_failure_mode'] = dict(result_dict)

In [60]:
len(np.unique(list(result_dict.keys()))), len(np.unique([val for item in result_dict.values() for val in item])), np.mean([len(val) for val in result_dict.values()])

(44, 6, 1.0)

## 7. Failure mode to sensor (FM2S) & 8. sensor to failure mode (S2FM)

In [61]:
fname = '../data/raw/iso_sensors.xlsx'

In [62]:
sheets = pd.ExcelFile(fname).sheet_names

In [63]:
dfs = {}
for sheet in sheets[1:]:
    dfs[sheet] = pd.read_excel(fname, sheet_name=sheet)

In [64]:
dfs['centrifugal pump'] = dfs.pop('pump')
dfs['centrifugal compressor'] = dfs.pop('compressor')
dfs['centrifugal fan'] = dfs.pop('fan')

In [65]:
try:
    dfs['reciprocating internal combustion'] = dfs.pop('reciprocating internal combusti')
except:
    pass

In [66]:
fault_replace = {
    'Eccentric rotor fault': 'Eccentric rotor',
    'Bearing wear/damage': 'Bearing wear',
    'Bearing damage': 'Bearing wear',
    'Power turbine damage': 'Power turbine damaged',
    'reciprocating internal combusti': 'reciprocating internal combustion',
    'Supply faults, e.g. excessive harmonics and over fluxing': 'Supply faults e.g. excessive harmonics and over fluxing'
}
sensor_replace = {
    'volts': 'voltage',
    'coast down': 'coast down time',
    'compresor pressure': 'compressor pressure',
    'consumption': 'oil consumption',
    'contamination': 'oil debris contamination',
    'dielecric frequency response (dfr)': 'dielectric frequency response',
    'frequency response analysis (fra)': 'frequency response analysis',
    'recovery_voltage_method_(rvm)': 'recovery_voltage_method',
    'polarization and de-polarization current (pdc)': 'polarization and de-polarization current',
    'power factor': 'power factor tan delta',
    'temparature': 'temperature'
    
}
asset_categories = {
    'electric motor': 'electric', 
    'steam turbine': 'rotating',
    'aero gas turbine': 'rotating',
    'industrial gas turbine': 'rotating',
    'pump': 'mechanical',
    'centrifugal pump': 'mechanical',
    'compressor': 'mechanical',
    'centrifugal compressor': 'mechanical',
    'electric generator': 'electric',
    'centrifugal fan': 'rotating',
    'power transformer': 'electric',
    'reciprocating internal combustion': 'mechanical',
    
}
drop_sensors = ['tanδ', 'pressure ratio']

In [67]:
def clean_items(arr):
    parsed_items = []
    for item in arr:
        parsed_items.extend(item.split('/'))
        for i, item in enumerate(parsed_items):
            if item in sensor_replace:
                parsed_items[i] = sensor_replace[item]
        for item in drop_sensors:
            if item in parsed_items:
                parsed_items.remove(item)
    return parsed_items

In [68]:
records = []
all_positives = []
all_negatives = []
all_faults = []

for asset in dfs:
    asset_df = dfs[asset]
    for _, row in asset_df.iterrows():
        keys = row.keys()
        fault = row[keys[0]]
        fault = re.sub(r'(\s*)*\/(\s*)*', '/', fault)
        if fault in fault_replace:
            fault = fault_replace[fault]
        positives = [key for key in keys[1:] if row[key] == 'X']
        negatives = [key for key in keys[1:] if row[key] != 'X']
        # remove spaces between or after "/"
        positives = [re.sub(r'(\s*)*\/(\s*)*', '/', text).lower() for text in positives]
        negatives = [re.sub(r'(\s*)*\/(\s*)*', '/', text).lower() for text in negatives]
        positives = clean_items(positives)
        negatives = clean_items(negatives)
        all_faults.append(fault.lower())
        all_positives.extend(positives)
        all_negatives.extend(negatives)
        records.append({'asset': asset.lower(), 'fault': fault.lower(), 'positive': positives, 'negative': negatives})

In [69]:
instruction = 'For a given asset, its category, and a sensor what are the faults that can be detected?'

In [72]:
with open("../data/industrial/synthetic_desc/sensors.yaml") as stream:
    sensor_desc = yaml.safe_load(stream)['sensors']
with open("../data/industrial/synthetic_desc/assets.yaml") as stream:
    asset_desc = yaml.safe_load(stream)['industrial_assets']
all_asset_descs = {list(asset.keys())[0].lower(): list(asset.values())[0] for eq_type in asset_desc for asset in asset_desc[eq_type]['assets']}

In [73]:
fault_descs_lower = {}
for key in other_descs['fault']:
   fault_descs_lower[key.lower()] = other_descs['fault'][key]

In [74]:
# asset, fault, related/unrelated sensor
asset_sensors = defaultdict(list)
# asset, sensor, related/unrelated fault
asset_faults = defaultdict(list)
for item in records:
    instruction = random.choice(failure_mode_to_sensor)
    query = f'Instruct: {instruction}\nQuery: Asset: {item["asset"]}, <asset_desc>Category: {asset_categories[item["asset"]]}, Fault: {item["fault"]}'
    asset_str = f'Asset Description: {all_asset_descs[item["asset"]]}, ' if random.random() < p_desc else ''
    query = query.replace('<asset_desc>', asset_str)
    if random.random() < p_desc:
        query += ', Fault Description: ' + fault_descs_lower[item['fault']]
    asset_sensors[query].extend([f"Sensor: {x}" for x in item['positive']])
    for sensor in item['positive']:
        instruction = random.choice(sensor_to_failure_mode)
        query = f'Instruct: {instruction}\nQuery: Asset: {item["asset"]}, <asset_desc>Category: {asset_categories[item["asset"]]}, Sensor: {sensor} <sensor_desc>'
        # with random probability add a sensor description to the query
        asset_str = ''
        sensor_str = ''
        if random.random() < p_desc:
            asset_str = f'Asset Description: {all_asset_descs[item["asset"]]}, '
        if random.random() < p_desc:
            sensor_formatted = sensor.lower().replace('sensor: ', '').replace(' ', '_')
            sensor_str = f', Sensor Description: {sensor_desc[sensor_formatted]}'
        query = query.replace('<asset_desc>', asset_str).replace('<sensor_desc>', sensor_str)
        asset_faults[query].append(f"Fault: {item['fault']}")

In [75]:
unique_sensors = np.unique([sensor for asset in asset_sensors for sensor in asset_sensors[asset]])
unique_assets = np.unique([asset for asset in asset_sensors])

In [76]:
dfs.keys()

dict_keys(['electric motor', 'steam turbine', 'aero gas turbine', 'industrial gas turbine', 'electric generator', 'power transformer', 'centrifugal pump', 'centrifugal compressor', 'centrifugal fan', 'reciprocating internal combustion'])

In [77]:
key = list(asset_sensors.keys())[0]
key, asset_sensors[key]

('Instruct: What sensors are capable of detecting the fault in the given asset and category?\nQuery: Asset: electric motor, Asset Description: Converts electrical energy into mechanical energy to power various industrial machinery., Category: electric, Fault: rotor windings fault',
 ['Sensor: current',
  'Sensor: power',
  'Sensor: torque',
  'Sensor: speed',
  'Sensor: vibration',
  'Sensor: temperature',
  'Sensor: axial flux',
  'Sensor: cooling gas'])

In [78]:
key = list(asset_faults.keys())[1]
key, asset_faults[key]

('Instruct: What faults can a sensor pick up in a given asset and category?\nQuery: Asset: electric motor, Category: electric, Sensor: power , Sensor Description: General sensor for measuring electrical power in systems to ensure optimal power usage and early detection of failures.',
 ['Fault: rotor windings fault'])

In [79]:
all_dicts['asset_fault_to_sensor'] = dict(asset_sensors)
all_dicts['asset_sensor_to_fault'] = dict(asset_faults)

In [80]:
asset_faults[list(asset_faults.keys())[0]]

['Fault: rotor windings fault']

In [81]:
len(np.unique(list(asset_sensors.keys()))), len(np.unique([val for item in asset_sensors.values() for val in item])), np.mean([len(val) for val in asset_sensors.values()])

(111, 53, 4.513513513513513)

In [82]:
len(np.unique(list(asset_faults.keys()))), len(np.unique([val for item in asset_faults.values() for val in item])), np.mean([len(val) for val in asset_faults.values()])

(485, 55, 1.0329896907216496)

## 9. Asset to related sensors

In [83]:
instruction = 'For a given asset and asset class what sensors are relevant?'

In [84]:
asset_records = {}
for record in records:
    asset_records[record['asset']] = record['positive'] + record['negative']

In [85]:
len(asset_records.keys())

10

In [86]:
result_dict = {}
for asset in asset_records:
    instruction = random.choice(asset_to_sensors)
    query = f"Instruct: {instruction}\nQuery: Asset: {asset}, <asset_desc>Category: {asset_categories[asset]}"
    asset_str = f'Asset Description: {all_asset_descs[record["asset"]]}, ' if random.random() < p_desc else ''
    query = query.replace('<asset_desc>', asset_str)
    # all positive/negative sensors have at least one related fault for that asset
    result_dict[query] = [f"Sensor: {sensor}" for sensor in asset_records[asset]]

In [87]:
key = list(result_dict.keys())[2]
key, result_dict[key]

('Instruct: Given an asset and its class, which sensors are most applicable?\nQuery: Asset: aero gas turbine, Asset Description: An engine that converts chemical energy from fuel into mechanical energy through piston movement., Category: rotating',
 ['Sensor: vibration',
  'Sensor: compressor temperature',
  'Sensor: compressor pressure',
  'Sensor: air flow',
  'Sensor: fuel pressure',
  'Sensor: fuel flow',
  'Sensor: speed',
  'Sensor: gas generator temperature',
  'Sensor: pressure',
  'Sensor: power turbine temperature',
  'Sensor: exhaust temperature',
  'Sensor: oil debris',
  'Sensor: oil leakage',
  'Sensor: oil consumption'])

In [88]:
all_dicts['asset_to_related_sensors'] = result_dict.copy()

In [89]:
len(np.unique(list(result_dict.keys()))), len(np.unique([val for item in result_dict.values() for val in item])), np.mean([len(val) for val in result_dict.values()])

(10, 53, 12.6)

## Build all queries, corpus and relations

In [90]:
all_queries = []
all_corpus = []

In [91]:
for dataset_name in all_dicts:
    dataset = all_dicts[dataset_name]
    all_queries.extend(list(dataset.keys()))
    for query in dataset:
        if type(dataset[query]) != list:
            print(dataset_name, query, dataset[query])
        all_corpus.extend(dataset[query])

In [92]:
all_queries = np.unique(all_queries).tolist()
query_to_id = dict(zip(all_queries, np.arange(len(all_queries)).tolist()))
all_corpus = np.unique(all_corpus).tolist()
corpus_to_id = dict(zip(all_corpus, np.arange(len(all_corpus)).tolist()))

In [93]:
query_to_doc = {}
dataset_to_queries = {}
for dataset_name in all_dicts:
    dataset_to_queries[dataset_name] = []
    dataset = all_dicts[dataset_name]
    for query in dataset:
        qid = query_to_id[query]
        dataset_to_queries[dataset_name].append(qid)
        docs = dataset[query]
        if qid not in query_to_doc:
            query_to_doc[qid] = set()
        for doc in docs:
            doc_id = corpus_to_id[doc]
            query_to_doc[qid].add(doc_id)

In [94]:
for key in query_to_doc:
    query_to_doc[key] = list(query_to_doc[key])

In [95]:
len(all_queries), len(all_corpus)

(1139, 1674)

In [96]:
out_dict = {
    'query_to_id': query_to_id,
    'corpus_to_id': corpus_to_id,
    'query_to_doc': query_to_doc,
    'dataset_to_queries': dataset_to_queries
}

In [85]:
#todo: check some keys are splitted chars

In [86]:
# for item in all_dicts:
#     print("*"*90)
#     print(f"Task: {item}, num_queries: {len(all_dicts[item])}")
#     first_q = list(all_dicts[item].keys())[0]
#     first_a = all_dicts[item][first_q]
#     print(first_q)
#     print(first_a)

### Construct train/validation/test datasets

In [97]:
id_to_query = dict(zip(out_dict['query_to_id'].values(), out_dict['query_to_id'].keys()))
id_to_corpus = dict(zip(out_dict['corpus_to_id'].values(), out_dict['corpus_to_id'].keys()))
query_to_doc = out_dict['query_to_doc']
for key in query_to_doc:
    query_to_doc[key] = set(query_to_doc[key])

In [98]:
# for the datasets that need splitting by asset
get_asset = lambda x: re.search('Asset: (.)*, Category', x).group(0).replace('Asset: ', '').replace(', Category', '')
assets = np.unique([get_asset(id_to_query[qid]) for qid in out_dict['dataset_to_queries']['asset_sensor_to_fault']]).tolist()
train_assets, test_assets = train_test_split(assets, train_size=0.8)
train_assets, val_assets = train_test_split(train_assets, train_size=0.75)

## Split randomly by questions for tasks that it makes sense and for others split by asset

In [99]:
train_datasets = {}
eval_datasets = {}
test_datasets = {}
for ds_name in out_dict['dataset_to_queries']:
    # 60/20/20 split
    if ds_name in ['asset_sensor_to_fault', 'asset_fault_to_sensor']:
        # will be split by asset name
        ds_queries = [id_to_query[qid] for qid in out_dict['dataset_to_queries'][ds_name]]
        ds_qids = out_dict['dataset_to_queries'][ds_name]
        assets = [get_asset(query) for query in ds_queries]
        train_qids = [qid for i, qid, asset in zip(np.arange(len(ds_qids)), ds_qids, assets) if asset in train_assets]
        val_qids = [qid for i, qid, asset in zip(np.arange(len(ds_qids)), ds_qids, assets) if asset in val_assets]
        test_qids = [qid for i, qid, asset in zip(np.arange(len(ds_qids)), ds_qids, assets) if asset in test_assets]
    else:
        ds_qids = out_dict['dataset_to_queries'][ds_name]
        train_qids, test_qids = train_test_split(ds_qids, train_size=0.8)
        train_qids, val_qids = train_test_split(train_qids, train_size=0.75)
    # note: we use all possible docs as negatives (even the ones in validation and test set)
    all_docs = set([id_to_corpus[x] for qid in ds_qids for x in query_to_doc[qid]])
    train_queries = [id_to_query[qid] for qid in train_qids]
    train_corpus_pos = [[id_to_corpus[rel_qid] for rel_qid in query_to_doc[qid]] for qid in train_qids]
    train_corpus_neg = [list(all_docs - set(x)) for x in train_corpus_pos]
    val_queries = [id_to_query[qid] for qid in val_qids]
    val_corpus_pos = [[id_to_corpus[rel_qid] for rel_qid in query_to_doc[qid]] for qid in val_qids]
    val_corpus_neg = [list(all_docs - set(x)) for x in val_corpus_pos]
    test_queries = [id_to_query[qid] for qid in test_qids]
    test_corpus_pos = [[id_to_corpus[rel_qid] for rel_qid in query_to_doc[qid]] for qid in test_qids]
    test_corpus_neg = [list(all_docs - set(x)) for x in test_corpus_pos]
    # todo: create the dataset
    train_dataset = [{'sentence1': train_queries[i], 'sentence2': doc, 'label': 1} for i, item in enumerate(train_corpus_pos) for doc in item]
    train_dataset += [{'sentence1': train_queries[i], 'sentence2': doc, 'label': 0} for i, item in enumerate(train_corpus_neg) for doc in item]
    val_dataset = [{'sentence1': train_queries[i], 'sentence2': doc, 'label': 1} for i, item in enumerate(val_corpus_pos) for doc in item]
    val_dataset += [{'sentence1': train_queries[i], 'sentence2': doc, 'label': 0} for i, item in enumerate(val_corpus_neg) for doc in item]
    test_dataset = [{'sentence1': train_queries[i], 'sentence2': doc, 'label': 1} for i, item in enumerate(test_corpus_pos) for doc in item]
    test_dataset += [{'sentence1': train_queries[i], 'sentence2': doc, 'label': 0} for i, item in enumerate(test_corpus_neg) for doc in item]
    train_datasets[ds_name] = train_dataset
    eval_datasets[ds_name] = val_dataset
    test_datasets[ds_name] = test_dataset

In [100]:
instructions = {
    'eq_to_category': equipment_category_to_equipments,
    'eq_to_class_type': equipment_class_to_type,
    'eq_subunit_to_unit': equipment_subunit_to_unit,
    'failure_desc_to_class': failure_description_to_failure_class,
    'asset_fm_to_components': asset_class_failure_mode_class_to_component,
    'component_to_failure_mode': component_to_failure_mode,
    'asset_fault_to_sensor': failure_mode_to_sensor,
    'asset_sensor_to_fault': sensor_to_failure_mode,
    'asset_to_related_sensors': asset_to_sensors
}

## Split rephrases in train/validation/test splits

In [101]:
rephrase_instruct_splits = {'train': {}, 'val': {}, 'test': {}}
# split evenly the rephrases
for ds_name in instructions:
    rephrase_instruct_splits['train'][ds_name], rephrase_instruct_splits['test'][ds_name] = train_test_split(instructions[ds_name], test_size=1/3)
for ds_name in instructions:
    rephrase_instruct_splits['train'][ds_name], rephrase_instruct_splits['val'][ds_name] = train_test_split(instructions[ds_name], test_size=1/2)

## Add random rephrasing and update the qids and qids to docs

In [102]:
# number of times to rephrase
n_rephrases = 2

In [103]:
# empty, because we'll rephrase
for task in out_dict['dataset_to_queries']:
    out_dict['dataset_to_queries'][task] = []

In [104]:
# add random rephrasing
regex_repl = 'Instruct: (.)*\nQuery'
for split_name, ds_split in zip(['train', 'val', 'test'], [train_datasets, eval_datasets, test_datasets]):
    for task, ds in ds_split.items():
        split_instructions = rephrase_instruct_splits[split_name][task]
        ds_split[task] = []
        task_qids = []
        # empty dataset to queries as we're gonna rephrase
        for _ in range(n_rephrases):
            for item in ds:
                # test, add new queries, and relationships with docs
                qid = out_dict['query_to_id'][item['sentence1']]
                corpus_ids = out_dict['query_to_doc'][qid]
                new_qid = qid + len(out_dict['query_to_id'])
                out_dict['query_to_doc'][new_qid] = out_dict['query_to_doc'][qid]
                # end test
                item['sentence1'] = re.sub(regex_repl, f'Instruct: {random.choice(split_instructions)}\nQuery', item['sentence1'])
                item['task'] = task
                # test
                out_dict['dataset_to_queries'][task].append(item['sentence1'])
                out_dict['query_to_id'][item['sentence1']] = new_qid
                # end test
                ds_split[task].append(item.copy())

In [105]:
all_train = [item for task, ds in train_datasets.items() for item in ds]
all_val = [item for task, ds in eval_datasets.items() for item in ds]
all_test = [item for task, ds in test_datasets.items() for item in ds]

In [106]:
all_data = all_train + all_val + all_test

In [107]:
all_queries = np.unique([item['sentence1'] for item in all_data])
all_queries = dict(zip(np.arange(len(all_queries)), all_queries))
all_docs = np.unique([item['sentence2'] for item in all_data])
all_docs = dict(zip(np.arange(len(all_docs)), all_docs))

In [113]:
out_path = '../data/industrial/processed'
destination_folder = out_path + f'/ablation/p_desc_{p_desc}_seed_{seed_no}'
if not os.path.exists(destination_folder):
    os.mkdir(destination_folder)

In [114]:
for split_name, ds in zip(['train', 'val', 'test'], [all_train, all_val, all_test]):
    ds = Dataset.from_list(ds)
    ds.to_json(destination_folder + f'/{split_name}.json')

Creating json from Arrow format:   0%|          | 0/130 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/51 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/45 [00:00<?, ?ba/s]

In [115]:
for key in out_dict['query_to_doc']:
    out_dict['query_to_doc'][key] = list(out_dict['query_to_doc'][key])

In [117]:
with open(f'{destination_folder}/all_tasks_dataset.json', 'w') as f:
    f.write(json.dumps(out_dict))